# Notebook 05: Unresolved Violation Classification

## Purpose

Train violation-level classification models that estimate whether a Boston housing violation case is more likely to remain unresolved.

This notebook compares simple baselines with supervised models:

1. Overall unresolved-rate baseline
2. Code-only baseline
3. Logistic regression
4. Random forest

The output is interpreted as a prioritization score, not a perfect yes/no prediction.

In [1]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

features_path = PROCESSED_DIR / 'violation_level_model_features.csv'
metadata_path = PROCESSED_DIR / 'violation_level_feature_columns.json'

if not features_path.exists():
    raise FileNotFoundError(f'Missing feature table: {features_path}. Run Notebook 04 first.')
if not metadata_path.exists():
    raise FileNotFoundError(f'Missing feature metadata: {metadata_path}. Run Notebook 04 first.')

model_df = pd.read_csv(features_path, low_memory=False)
metadata = json.loads(metadata_path.read_text())

target_col = metadata['target']
numeric_features = [col for col in metadata['numeric_features'] if col in model_df.columns]
categorical_features = [col for col in metadata['categorical_features'] if col in model_df.columns]

model_df = model_df.dropna(subset=[target_col]).copy()
model_df[target_col] = model_df[target_col].astype(int)
for col in categorical_features:
    model_df[col] = model_df[col].fillna('Missing').astype(str)

print(f'Rows: {len(model_df):,}')
print(f'Unresolved rate: {model_df[target_col].mean():.2%}')
print(f'Numeric features: {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')

Rows: 16,760
Unresolved rate: 4.74%
Numeric features: 21
Categorical features: 9


## 1. Train/Test Split

We use a stratified train/test split because unresolved cases are rare. Accuracy is not the main metric; top-decile lift, ROC-AUC, and average precision are more informative for prioritization.

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.25
TOP_FRACTION = 0.10

required_cols = [target_col] + numeric_features + categorical_features
train_df, test_df = train_test_split(
    model_df[required_cols],
    test_size=TEST_SIZE,
    stratify=model_df[target_col],
    random_state=RANDOM_STATE,
)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].to_numpy()
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].to_numpy()

print(f'Train rows: {len(train_df):,}')
print(f'Test rows:  {len(test_df):,}')
print(f'Test unresolved rate: {y_test.mean():.2%}')

Train rows: 12,570
Test rows:  4,190
Test unresolved rate: 4.73%


## 2. Baselines and Models

The code-only baseline directly answers the concern that people may already infer risk from violation code. The supervised models test whether combining violation type with location and neighborhood context improves prioritization.

In [3]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def make_preprocessor(scale_numeric=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))

    transformers = []
    if numeric_features:
        transformers.append(('numeric', Pipeline(numeric_steps), numeric_features))
    if categorical_features:
        transformers.append(('categorical', make_one_hot_encoder(), categorical_features))
    return ColumnTransformer(transformers=transformers, remainder='drop')


models = {
    'Logistic Regression': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=True)),
        ('model', LogisticRegression(max_iter=2500, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
    'Random Forest': Pipeline([
        ('preprocess', make_preprocessor(scale_numeric=False)),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=18,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
}

print('Models:', list(models))

Models: ['Logistic Regression', 'Random Forest']


## 3. Evaluation Helpers

In [4]:
def top_decile_metrics(y_true, risk_score, top_fraction=TOP_FRACTION):
    frame = pd.DataFrame({'actual': y_true.astype(int), 'risk': risk_score.astype(float)})
    top_n = max(1, int(np.ceil(len(frame) * top_fraction)))
    top = frame.sort_values('risk', ascending=False).head(top_n)

    overall_rate = frame['actual'].mean()
    top_rate = top['actual'].mean()
    captured_share = top['actual'].sum() / frame['actual'].sum() if frame['actual'].sum() else np.nan

    return {
        'top_10_pct_cases': float(top_n),
        'overall_unresolved_rate': float(overall_rate),
        'top_10_pct_unresolved_rate': float(top_rate),
        'top_10_pct_lift': float(top_rate / overall_rate) if overall_rate > 0 else np.nan,
        'captured_unresolved_share': float(captured_share),
    }


def evaluate(model_name, y_true, risk_score):
    pred = (risk_score >= 0.5).astype(int)
    out = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, risk_score),
        'average_precision': average_precision_score(y_true, risk_score),
    }
    out.update(top_decile_metrics(y_true, risk_score))
    return out


def code_only_risk(train_frame, test_frame, code_col='code_grouped'):
    global_rate = train_frame[target_col].mean()
    code_rates = train_frame.groupby(code_col)[target_col].mean()
    return test_frame[code_col].map(code_rates).fillna(global_rate).to_numpy()

## 4. Train Models

In [ ]:
metric_rows = []
prediction_frames = []
fitted_models = {}

# Baseline 1: same risk for every case.
overall_risk = np.repeat(y_train.mean(), len(y_test))
metric_rows.append(evaluate('Overall Rate Baseline', y_test, overall_risk))
prediction_frames.append(pd.DataFrame({
    'model': 'Overall Rate Baseline',
    'actual_is_unresolved': y_test,
    'predicted_unresolved_risk': overall_risk,
}))

# Baseline 2: historical unresolved rate by violation code group.
code_risk = code_only_risk(train_df, test_df)
metric_rows.append(evaluate('Code-Only Baseline', y_test, code_risk))
prediction_frames.append(pd.DataFrame({
    'model': 'Code-Only Baseline',
    'actual_is_unresolved': y_test,
    'predicted_unresolved_risk': code_risk,
}))

# Supervised models.
for name, pipeline in models.items():
    print(f'Training {name}...')
    fitted = pipeline.fit(X_train, y_train)
    fitted_models[name] = fitted
    risk = fitted.predict_proba(X_test)[:, 1]

    metric_rows.append(evaluate(name, y_test, risk))
    prediction_frames.append(pd.DataFrame({
        'model': name,
        'actual_is_unresolved': y_test,
        'predicted_unresolved_risk': risk,
    }))

metrics_df = pd.DataFrame(metric_rows).sort_values('top_10_pct_lift', ascending=False).reset_index(drop=True)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

Training Logistic Regression...
Training Random Forest...


,model,accuracy,precision,recall,f1,roc_auc,average_precision,top_10_pct_cases,overall_unresolved_rate,top_10_pct_unresolved_rate,top_10_pct_lift,captured_unresolved_share
0,Logistic Regression,0.724821,0.115230,0.722222,0.198749,0.789682,0.133602,419.0,0.047255,0.152745,3.232323,0.323232
1,Random Forest,0.805489,0.126061,0.525253,0.203324,0.761166,0.133880,419.0,0.047255,0.145585,3.080808,0.308081
2,Code-Only Baseline,0.952745,0.000000,0.000000,0.000000,0.740331,0.098151,419.0,0.047255,0.116945,2.474747,0.247475
3,Overall Rate Baseline,0.952745,0.000000,0.000000,0.000000,0.500000,0.047255,419.0,0.047255,0.035800,0.757576,0.075758


## 5. Save Metrics and Figures

In [6]:
metrics_path = PROCESSED_DIR / 'classification_model_metrics.csv'
predictions_path = PROCESSED_DIR / 'classification_model_predictions.csv'
summary_path = PROCESSED_DIR / 'classification_prioritization_summary.csv'

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(predictions_path, index=False)

best_model_name = metrics_df[metrics_df['model'].isin(['Logistic Regression', 'Random Forest'])].iloc[0]['model']
best_summary = metrics_df[metrics_df['model'] == best_model_name][[
    'top_10_pct_cases',
    'overall_unresolved_rate',
    'top_10_pct_unresolved_rate',
    'top_10_pct_lift',
    'captured_unresolved_share',
]].T.reset_index()
best_summary.columns = ['metric', 'value']
best_summary.to_csv(summary_path, index=False)

print(f'Saved metrics -> {metrics_path}')
print(f'Saved predictions -> {predictions_path}')
print(f'Saved prioritization summary -> {summary_path}')
print(f'Best supervised model by top-10% lift: {best_model_name}')

Saved metrics -> D:\Projects\CS506Project\data\processed\classification_model_metrics.csv
Saved predictions -> D:\Projects\CS506Project\data\processed\classification_model_predictions.csv
Saved prioritization summary -> D:\Projects\CS506Project\data\processed\classification_prioritization_summary.csv
Best supervised model by top-10% lift: Logistic Regression


In [7]:
plot_metrics = metrics_df.melt(
    id_vars='model',
    value_vars=['precision', 'recall', 'f1', 'roc_auc', 'average_precision', 'top_10_pct_lift'],
    var_name='metric',
    value_name='score',
)
fig, ax = plt.subplots(figsize=(11, 5.8))
sns.barplot(data=plot_metrics, x='score', y='metric', hue='model', ax=ax)
ax.set_xlabel('Score')
ax.set_ylabel('')
ax.set_title('Unresolved-Case Prioritization Model Comparison')
ax.legend(title='Model', loc='lower right')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '07_classification_model_comparison.png', bbox_inches='tight')
plt.close(fig)

best_predictions = predictions_df[predictions_df['model'] == best_model_name]
y_best = best_predictions['actual_is_unresolved'].astype(int).to_numpy()
risk_best = best_predictions['predicted_unresolved_risk'].to_numpy()
cm = confusion_matrix(y_best, (risk_best >= 0.5).astype(int), labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.8, 5.2))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    xticklabels=['Pred resolved', 'Pred unresolved'],
    yticklabels=['Actual resolved', 'Actual unresolved'],
    ax=ax,
)
ax.set_title(f'Confusion Matrix at 0.50 Threshold: {best_model_name}')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '08_classification_confusion_matrix.png', bbox_inches='tight')
plt.close(fig)

lift_df = metrics_df[['model', 'top_10_pct_lift']].sort_values('top_10_pct_lift', ascending=False)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
sns.barplot(data=lift_df, x='top_10_pct_lift', y='model', ax=ax)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Top-10% Lift Over Overall Unresolved Rate')
ax.set_ylabel('')
ax.set_title('Prioritization Lift')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '09_prioritization_lift.png', bbox_inches='tight')
plt.close(fig)

print('Saved model figures 07-09.')

Saved model figures 07-09.


## 6. Feature Importance

In [8]:
def get_feature_names(preprocessor):
    names = list(numeric_features)
    if 'categorical' in preprocessor.named_transformers_:
        onehot = preprocessor.named_transformers_['categorical']
        names.extend(onehot.get_feature_names_out().tolist())
    return names

if 'Random Forest' in fitted_models:
    rf = fitted_models['Random Forest']
    names = get_feature_names(rf.named_steps['preprocess'])
    importances = rf.named_steps['model'].feature_importances_
    importance_df = (
        pd.DataFrame({'feature': names, 'importance': importances})
        .sort_values('importance', ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(9, 6.2))
    sns.barplot(data=importance_df, x='importance', y='feature', ax=ax)
    ax.set_xlabel('Random Forest Importance')
    ax.set_ylabel('')
    ax.set_title('Top Feature Importances')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '10_feature_importance.png', bbox_inches='tight')
    plt.close(fig)
    display(importance_df)
else:
    print('Random Forest was not trained; skipping feature importance.')

,feature,importance
0,latitude,0.113618
1,longitude,0.110758
50,code_grouped_116.2,0.081450
32,code_grouped_105.1,0.061455
48,code_grouped_116,0.027935
27,code_grouped_102.8,0.026157
9,median_living_area,0.022776
19,rentsmart_share_condo,0.021357
12,share_multifamily,0.019956
11,share_condo,0.018899


## Summary for Report

The full Random Forest model is compared against a code-only baseline. If the full model improves top-decile lift, that supports the claim that location and neighborhood context add useful prioritization information beyond violation code alone. If the improvement is small, that is still a meaningful result: violation code already captures much of the unresolved-risk signal.